# AICE-Style Practice: Seoul Food Delivery (Regression)
이 문제세트는 공개 AICE 샘플과 **유사한 흐름**으로 구성되었지만, **완전히 새로운 데이터셋**을 사용합니다.

### 데이터
- `food_delivery_seoul.csv` : 주문 단위의 배달 기록
- `traffic_events.csv` : 동일 `OrderID` 기준의 교차 신호등 수

### 목표(Target)
- `Delivery_Time_Minutes` (분) 예측

### 권장 절차(문항 1~14)
임포트 → 읽기/머지 → 시각화(countplot, jointplot) → 이상치 제거 → 결측 처리 → 불필요 컬럼 삭제 → 원-핫 인코딩 → 스케일링 → 의사결정나무/랜덤포레스트 → MAE 비교 → 간단 MLP 회귀 → 학습곡선 시각화


## 1) scikit-learn 임포트

In [ ]:
## 여기에 답안코드를 작성하세요.
import sklearn as sk

## 2) pandas 임포트

In [ ]:
## 여기에 답안코드를 작성하세요.
import pandas as pd

## 3) 데이터 읽고 merge (inner) → df

In [ ]:
df_main = pd.read_csv('food_delivery_seoul.csv')
df_traffic = pd.read_csv('traffic_events.csv')
df = pd.merge(df_main, df_traffic, on='OrderID', how='inner')
df.head()

In [ ]:
## 여기에 답안코드를 작성하세요.
df_main = pd.read_csv('food_delivery_seoul.csv')
df_traffic = pd.read_csv('traffic_events.csv')
df = pd.merge(df_main, df_traffic, on='OrderID', how='inner')
df.head()

다음 셀을 먼저 실행하세요(한글 폰트 지정)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
plt.rc('font', family='NanumGothicCoding')

## 4) Region1 분포 countplot + '-' 제거
- 그래프를 보고 4지선다 중 **틀린 보기 번호**를 `답안04`에 저장하세요 (예: `답안04 = 2`)

In [ ]:
## 여기에 답안코드를 작성하세요.
import seaborn as sns
sns.countplot(data=df, x='Region1')
plt.xticks(rotation=25)
plt.show()
# '-' 제거
df = df[df['Region1']!='-'].copy()
답안04 = None

## 5) jointplot: Distance_km vs Delivery_Time_Minutes

In [ ]:
## 여기에 답안코드를 작성하세요.
import seaborn as sns
sns.jointplot(data=df, x='Distance_km', y='Delivery_Time_Minutes')

## 6) 이상치 제거 + 'OrderID' 컬럼 삭제 → df_temp
- 기준: `Avg_Speed_kmh >= 120` 는 이상치로 간주하여 제거

In [ ]:
## 여기에 답안코드를 작성하세요.
df_temp = df[df['Avg_Speed_kmh'] < 120].copy()
df_temp = df_temp.drop(columns=['OrderID'])
df_temp.head()

## 7) 결측치 확인/제거 → df_na, 결측 총개수 `답안07` 저장

In [ ]:
## 여기에 답안코드를 작성하세요.
na_cnt = df_temp.isnull().sum().sum()
df_na = df_temp.dropna().copy()
답안07 = na_cnt
답안07

## 8) 불필요 시간 컬럼 삭제 → df_del
- `Time_Order_Placed`, `Time_Order_Delivered`

In [ ]:
## 여기에 답안코드를 작성하세요.
to_drop = ['Time_Order_Placed','Time_Order_Delivered']
df_del = df_na.drop(columns=to_drop, errors='ignore').copy()
df_del.head()

## 9) 범주형 전체 원-핫 인코딩 → df_preset

In [ ]:
## 여기에 답안코드를 작성하세요.
obj_cols = df_del.select_dtypes(include='object').columns.tolist()
df_preset = pd.get_dummies(df_del, columns=obj_cols)
df_preset.head()

## 10) train/valid 분리 + RobustScaler

In [ ]:
## 여기에 답안코드를 작성하세요.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
X = df_preset.drop('Delivery_Time_Minutes', axis=1).values
y = df_preset['Delivery_Time_Minutes'].values
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
rs = RobustScaler()
X_train = rs.fit_transform(X_train)
X_valid = rs.transform(X_valid)

## 11) DecisionTree & RandomForest 학습

In [ ]:
## 여기에 답안코드를 작성하세요.
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
dt = DecisionTreeRegressor(max_depth=6, min_samples_split=4, random_state=120)
rf = RandomForestRegressor(max_depth=6, min_samples_split=4, random_state=120)
dt.fit(X_train, y_train); rf.fit(X_train, y_train)

## 12) MAE 비교 → `답안12` ('randomforest' 또는 'decisiontree')

In [ ]:
## 여기에 답안코드를 작성하세요.
from sklearn.metrics import mean_absolute_error
y_pred_dt = dt.predict(X_valid); y_pred_rf = rf.predict(X_valid)
dt_mae = mean_absolute_error(y_valid, y_pred_dt)
rf_mae = mean_absolute_error(y_valid, y_pred_rf)
답안12 = 'randomforest' if rf_mae < dt_mae else 'decisiontree'
dt_mae, rf_mae, 답안12

TensorFlow 셀을 먼저 실행하세요

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tf.random.set_seed(7)

## 13) 간단 MLP 회귀 학습 → `history` 저장

In [ ]:
## 여기에 답안코드를 작성하세요.
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'), Dropout(0.2),
    Dense(16, activation='relu'), Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mse'])
history = model.fit(X_train, y_train, epochs=25, batch_size=32,
                    validation_data=(X_valid, y_valid), verbose=0)
list(history.history.keys())

## 14) 학습/검증 MSE 시각화

In [ ]:
## 여기에 답안코드를 작성하세요.
import matplotlib.pyplot as plt
plt.plot(history.history['mse']); plt.plot(history.history['val_mse'])
plt.title('Model MSE'); plt.xlabel('Epochs'); plt.ylabel('MSE');
plt.legend(['mse','val_mse']); plt.show()